# The Scientific Stack - Overcoming Python's Bottlenecks

## Introduction: The "Glue Language" Paradigm
In the previous notebook, we identified two massive roadblocks for High-Performance Computing in pure Python:
1. **The Memory Overhead:** Python objects (`PyObject`) are bulky, and Python lists fragment memory, destroying CPU cache locality.
2. **The GIL:** The Global Interpreter Lock prevents multiple threads from executing Python bytecodes simultaneously.

If pure Python is poorly suited for heavy numerical analysis, why is it the dominant language in modern data science and computational physics? The answer lies in the **Scientific Stack** (primarily NumPy and SciPy). In HPC, Python acts merely as a "glue language." It handles the high-level logic, file I/O, and setup, but it outsources the actual mathematical heavy lifting to underlying libraries written in C, C++, and crucially, **Fortran**.

## 1. NumPy

NumPy (Numerical Python) is the foundation of the scientific stack. Its core object, the `ndarray` (N-dimensional array), solves the Python memory fragmentation issue. 

Unlike a Python `list` (which is an array of pointers to objects scattered in RAM), a NumPy array is a strictly typed, **contiguous block of memory**, exactly like a Fortran array.

Let's look at the syntax, specify explicit data types (like we would in Fortran), and observe how NumPy allows us to choose between C-style (row-major) and Fortran-style (column-major) memory layouts.

In [ ]:
import numpy as np
import sys

# Creating a standard Python list
py_list = [1.0, 2.0, 3.0, 4.0]

# Creating a strictly typed NumPy array (similar to real*8 in Fortran)
np_array = np.array([1.0, 2.0, 3.0, 4.0], dtype=np.float64)

print(f"Size of pure Python list (pointers only): {sys.getsizeof(py_list)} bytes")
print(f"Size of NumPy array (actual contiguous data): {sys.getsizeof(np_array)} bytes\n")

# Matrix Memory Layout: C vs Fortran
# By default, NumPy uses C-order (row-major). We can force Fortran-order (column-major).
matrix_c = np.ones((3, 3), dtype=np.float64, order='C')
matrix_f = np.ones((3, 3), dtype=np.float64, order='F')

print(f"Matrix C is C-contiguous: {matrix_c.flags['C_CONTIGUOUS']}")
print(f"Matrix F is Fortran-contiguous: {matrix_f.flags['F_CONTIGUOUS']}")

Size of pure Python list (pointers only): 88 bytes
Size of NumPy array (actual contiguous data): 144 bytes

Matrix C is C-contiguous: True
Matrix F is Fortran-contiguous: True


## 2. Vectorization: Bypassing the GIL and Python Loops

Because NumPy arrays are contiguous blocks of memory with a single data type, we no longer need to use Python `for` loops to iterate through them. 

Instead, we use Vectorization. Vectorized operations are pushed down to the compiled C/Fortran backend, which applies the operation to the entire array at once using CPU SIMD (Single Instruction, Multiple Data) instructions. This completely bypasses the Python interpreter and the GIL.**

Let's compare the performance of calculating the distance $d = \sqrt{x^2 + y^2}$ using a standard Python loop versus NumPy vectorization.

In [3]:
import time
import math
import numpy as np

N = 5_000_000

# Generating Data
x_list = list(range(N))
y_list = list(range(N))
x_np = np.arange(N, dtype=np.float64)
y_np = np.arange(N, dtype=np.float64)

# Pure Python approach (Slow: requires type-checking every iteration)
start_time = time.time()
distances_list = [math.sqrt(x_list[i]**2 + y_list[i]**2) for i in range(N)]
python_time = time.time() - start_time
print(f"Pure Python execution time:  {python_time:.4f} seconds")

# NumPy Vectorization (Fast: executed in compiled C code)
start_time = time.time()
distances_np = np.sqrt(x_np**2 + y_np**2)
numpy_time = time.time() - start_time
print(f"NumPy Vectorized execution:  {numpy_time:.4f} seconds")

speedup = python_time / numpy_time
print(f"\n--> NumPy Speedup Factor: {speedup:.1f}x")

Pure Python execution time:  8.3215 seconds
NumPy Vectorized execution:  0.1149 seconds

--> NumPy Speedup Factor: 72.4x



Closely related to vectorization is another fundamental NumPy feature: Broadcasting. 

Normally, when you perform math operations on two arrays, NumPy executes them element-by-element. In the strictest case, this requires both arrays to have the exact same shape.

In [4]:
# 1. Strict Element-by-Element operation (requires matching shapes)
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 2.0, 2.0])

result = a * b
print(f"Array * Array result: {result}")

Array * Array result: [2. 4. 6.]


Creating an entire array of identical numbers (like `b` above) just to scale another array is a waste of memory. 

NumPy’s broadcasting rule relaxes this constraint. The simplest example is operating between an array and a single scalar value.

In [5]:
# 2. Broadcasting an operation with a scalar
a = np.array([1.0, 2.0, 3.0])
scalar_b = 2.0

result_broadcast = a * scalar_b
print(f"Array * Scalar result: {result_broadcast}")

Array * Scalar result: [2. 4. 6.]


The output is identical. Conceptually, we can think of NumPy "stretches" the scalar `b` to match the shape of array `a` before doing the math. 

However, from an HPC perspective, this stretching is strictly virtual. NumPy is smart enough to use the original scalar value directly at the compiled C level. It does not allocate new memory or make physical copies of the scalar, making broadcasting operations efficient.

### 2.1 NumPy Linear Algebra
Because NumPy arrays are contiguous in memory, it is possible to handle them directly to highly optimized C and Fortran libraries (like BLAS and LAPACK) to do heavy matrix computations. Instead of writing messy nested for loops to multiply matrices or find eigenvalues, we can just use the numpy.linalg module.

Let's look at the basics: dot products, trace, determinants, matrix inversion, and eigenvalues.

In [6]:
A = np.array([[4.0, -2.0, 1.0], 
              [1.0,  5.0, 3.0], 
              [2.0,  1.0, 6.0]], dtype=np.float64)

B = np.array([[1.0, 0.0, 0.0], 
              [0.0, 1.0, 0.0], 
              [0.0, 0.0, 1.0]], dtype=np.float64) # Identity matrix

# Dot Product (Matrix Multiplication)
# In Python 3.5+, the '@' operator is preferred for matrix multiplication
C = A @ B 
print("Dot Product (A @ B):\n", C, "\n")

# Trace (Sum of diagonal elements)
trace_A = np.trace(A)
print(f"Trace of A: {trace_A}")

# Determinant
det_A = np.linalg.det(A)
print(f"Determinant of A: {det_A:.2f}")

# Matrix Inversion
inv_A = np.linalg.inv(A)
print("\nInverse of A:\n", inv_A)

# Eigenvalues and Eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(A)
print("\nEigenvalues of A:\n", eigenvalues)

Dot Product (A @ B):
 [[ 4. -2.  1.]
 [ 1.  5.  3.]
 [ 2.  1.  6.]] 

Trace of A: 15.0
Determinant of A: 99.00

Inverse of A:
 [[ 2.72727273e-01  1.31313131e-01 -1.11111111e-01]
 [-5.04646829e-18  2.22222222e-01 -1.11111111e-01]
 [-9.09090909e-02 -8.08080808e-02  2.22222222e-01]]

Eigenvalues of A:
 [6.26255113+0.8843676j 6.26255113-0.8843676j 2.47489775+0.j       ]


## 3. SciPy

While NumPy provides the data structures and basic operations, **SciPy** (Scientific Python) provides the advanced algorithms. SciPy is essentially a massive Python wrapper around highly optimized libraries written in Fortran and C (such as BLAS, LAPACK, MINPACK, and QUADPACK).

Instead of rewriting complex solvers in pure Python (which would be very slow), SciPy allows us to prepare the data in Python and pass it directly to a compiled Fortran routine under the hood.

Let's look at an example: Solving a large system of linear equations ($Ax = b$).

In [7]:
from scipy import linalg

# Create a large random coefficient matrix A and vector b
matrix_size = 2000
A = np.random.rand(matrix_size, matrix_size)
b = np.random.rand(matrix_size)

# Solving Ax = b
start_time = time.time()

# linalg.solve wraps optimized Fortran LAPACK routines (like dgesv)
# It automatically uses multi-threading and optimized CPU cache handling
x = linalg.solve(A, b) 

scipy_time = time.time() - start_time
print(f"Time to solve {matrix_size}x{matrix_size} system with SciPy: {scipy_time:.4f} seconds")

# Checking if the solution is correct (A*x - b should be near zero)
residual = np.linalg.norm(np.dot(A, x) - b)
print(f"Residual error: {residual:.2e}")

Time to solve 2000x2000 system with SciPy: 1.1932 seconds
Residual error: 1.39e-11


### 3.1 Sparse Matrices (`scipy.sparse`)

Real-world HPC problems (like fluid dynamics or finite element analysis), often deal with big matrices where the major part is constituted by zero elemets. Storing these as standard NumPy arrays wastes huge amounts of RAM and cache.

SciPy provides the `scipy.sparse` module to store only the non-zero elements. Let's see how much memory we can save, and how we can solve a sparse system very effectively.

In [1]:
import scipy.sparse as sparse
import scipy.sparse.linalg as splinalg
import sys
import time
import numpy as np

# SETUP
# We create a 5000 by 5000 grid.
N = 5000

# np.eye() creates an "identity matrix". It puts 1s on the diagonal and 0s everywhere else.
# Python is memorizing millions of useless zeros.
dense_matrix = np.eye(N)

# how much RAM is consuming? We transform it in megabyte.
dense_size = sys.getsizeof(dense_matrix) / (1024**2) 
print(f"Dense Matrix RAM usage:  {dense_size:.2f} MB")

# SciPy converts the dense matrix into a "Compressed Sparse Row" (CSR).
# Basically, SciPy stops memorizing the zeros and ONLY remembers where the actual numbers are.
sparse_matrix = sparse.csr_matrix(dense_matrix)

# Calculating the new RAM size. We have to add up the data arrays SciPy uses to keep track of things.
sparse_size = (sparse_matrix.data.nbytes + sparse_matrix.indptr.nbytes + sparse_matrix.indices.nbytes) / (1024**2)
print(f"Sparse Matrix RAM usage: {sparse_size:.4f} MB")
print(f"Memory saved: {dense_size / sparse_size:.0f}x\n")

# We create a random array of 5000 numbers to act as the 'b' in our equation $Ax = b$.
b_sparse = np.random.rand(N)

start_time = time.time()

# We use SciPy's special sparse solver. 
x_sparse = splinalg.spsolve(sparse_matrix, b_sparse)

print(f"Time to solve {N}x{N} sparse system: {time.time() - start_time:.4f} seconds")

Dense Matrix RAM usage:  190.73 MB
Sparse Matrix RAM usage: 0.0763 MB
Memory saved: 2500x

Time to solve 5000x5000 sparse system: 0.0070 seconds


### 3.2 Fast Fourier Transform (FFT)

Another useful feature of SciPy is signal processing. If you have a noisy signal or a combination of different waves, you can use the Fast Fourier Transform (FFT) to break it down and find the original frequencies. 

Here is a quick example of mixing two waves and using `scipy.fft` to extract their exact frequencies.

In [6]:
import numpy as np
from scipy.fft import fft, fftfreq

# We want to collect 800 samples (N) over 1 second
# T is the time gap between each measurement (1 second divided by 800)
N = 800
T = 1.0 / 800.0
# We create the timeline: an array going from 0 to 1 second with 800 steps in between
time = np.linspace(0.0, N*T, N, endpoint=False)

# Generate the first wave: traveling at 50 Hz (standard formula: sin(2 * pi * freq * time))
wave1 = np.sin(50.0 * 2.0 * np.pi * time)
# Generate the second wave: traveling at 120 Hz, but half the size (multiplied by 0.5)
wave2 = 0.5 * np.sin(120.0 * 2.0 * np.pi * time)
# Sum them together into one combined signal
combined_signal = wave1 + wave2

# fft() takes the messy signal and does the computation need to breaak it down
signal_fft = fft(combined_signal)
# fftfreq() calculates the X-axis values (it tells us which frequencies correspond to the extracted data).
frequencies = fftfreq(N, T)

# The FFT returns a mirrored spectrum (a positive half and a useless negative half).
# [:N//2] throws away the negative half so we only keep the needed one.
positive_freqs = frequencies[:N//2]

# The raw FFT results are "complex numbers" which are hard to read. 
# np.abs() extracts the absolute value, giving us a real number that represents the "strength" of the signal.
amplitudes = np.abs(signal_fft[0:N//2])

print("--- FFT Results ---")
print("Frequencies detected in the signal:")

# We can also apply a final filter
# We loop through all the frequencies the algorithm found...
for i in range(len(positive_freqs)):
    # ...but we ONLY print the ones that have a strong amplitude (greater than 100).
    # Without this 'if' statement, it would print a lot of near-zero frequencies caused by background math noise.
    if amplitudes[i] > 100: 
        print(f"Freq: {positive_freqs[i]} Hz")
        print(f"Ampl: {amplitudes[i]} dB")

    # print(f"Frequency: {positive_freqs[i]}")
    # print(f"Amplitude: {amplitudes[i]}")

--- FFT Results ---
Frequencies detected in the signal:
Freq: 50.0 Hz
Ampl: 400.000000000001 dB
Freq: 120.0 Hz
Ampl: 199.9999999999999 dB


## 4. Summary: The Scientific Stack in HPC

Before moving to distributed memory parallelism with MPI, let's summarize the Pros and Cons of using NumPy/SciPy compared to writing raw Fortran.

**Pros of Python + NumPy/SciPy:**
* **Development Speed:** Complex operations like solving matrices, interpolations, or Fourier transforms are reduced to a single line of code.
* **Ecosystem Integration:** Data can be generated by SciPy and instantly plotted with Matplotlib or saved to disk without complex I/O routines.
* **Efficiency:** You get the performance of decades-old Fortran libraries (BLAS/LAPACK) without having to manage Fortran pointers or module compilation.

**Cons and Limitations:**
* **The "Two-Language" Problem:** Debugging can be difficult. If a SciPy routine crashes deep inside its Fortran backend, Python might just throw a generic `Segmentation Fault` without a clear traceback.
* **Overhead for Small Tasks:** For massive arrays, NumPy is near-native speed. However, for executing millions of tiny, disconnected mathematical operations, the overhead of crossing the boundary between Python and C/Fortran becomes a bottleneck. In native Fortran, calling a tiny subroutine $10^6$ times has almost zero overhead.
* **No Auto-Parallelization of Custom Logic:** SciPy handles multithreading internally for matrix math. But if you have a custom, non-vectorizable physics algorithm, you are still restricted by Python's GIL. This forces the use of external parallelization tools (MPI), which we will explore in the next notebook.

## References

* NumPy documentation: [The N-dimensional array (ndarray)](https://numpy.org/doc/stable/reference/arrays.ndarray.html)
* NumPy documentation: [Memory Layout (C and Fortran order)](https://numpy.org/doc/stable/reference/arrays.ndarray.html#internal-memory-layout-of-an-ndarray)
* NumPy documentation: [Broadcasting and Vectorization](https://numpy.org/doc/stable/user/basics.broadcasting.html)
* SciPy documentation: [Linear Algebra (scipy.linalg)](https://docs.scipy.org/doc/scipy/reference/linalg.html)
* SciPy documentation: [scipy.linalg.solve (LAPACK wrappers)](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve.html)
